<a href="https://colab.research.google.com/github/EyadMHussien/A-Deep-Learning-Framework-for-Image-Captioning-Course-Advanced-Machine-Learning/blob/main/Milestone2_AML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- BLOCK 0: Connect Google Drive First! ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tf-keras

Project: A Deep Learning Framework for Image Captioning
Course: Advanced Machine Learning,

#Imports & Configuration

In [ ]:
# --- BLOCK 1: Imports & Configuration ---

import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm import tqdm
import zipfile
os.environ["TF_USE_LEGACY_KERAS"] = "1"
# TensorFlow/Keras imports
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint

# --- CONFIGURATION (Change these to tune your model) ---
CONFIG = {
    "DATA_DIR": "/content/data",
    "IMG_DIR": "data/train2014/train2014",
    "CAPTION_JSON": "data/captions/annotations/captions_train2014.json",
    "INSTANCES_JSON": "data/captions/annotations/instances_train2014.json",
    "FEATURES_FILE": "features.npy",
    "MAX_LENGTH": 35,
    "VOCAB_SIZE": 5000,
    "NUM_CLASSES": 80,
    "BATCH_SIZE": 64,
    "EPOCHS": 10
}

# Mount Drive for persistent saving
from google.colab import drive
drive.mount('/content/drive')
CONFIG["MODEL_SAVE_PATH"] = '/content/drive/MyDrive/coco_model_checkpoints/best_model.keras'
os.makedirs(os.path.dirname(CONFIG["MODEL_SAVE_PATH"]), exist_ok=True)

print("Block 1: Configuration loaded. Everything is modular and ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Block 1: Configuration loaded. Everything is modular and ready.


#Data Loading

In [ ]:
# --- BLOCK 2: Data Loading & Mapping (Google Drive Hardwired) ---
import os
import zipfile
import shutil
import json
import numpy as np

# Define where the zip will live safely in your Google Drive
DRIVE_ZIP_PATH = "/content/drive/MyDrive/AML_Project_Backup/ms-coco-dataset.zip"
LOCAL_ZIP_PATH = "ms-coco-dataset.zip"

# 1. Smart Download, Extract, and Backup
if not os.path.exists(CONFIG["DATA_DIR"]):
    print("Local data not found. Checking Google Drive...")

    # Scenario A: The Zip is already safely in your Drive!
    if os.path.exists(DRIVE_ZIP_PATH):
        print("Found dataset Zip in Google Drive! Copying to fast local workspace...")
        shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)

        print("Extracting dataset...")
        with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall('data')

    # Scenario B: We need to download it and back it up for the first time
    else:
        print("Dataset not found in Drive. Downloading from Kaggle...")
        !kaggle datasets download -d hariwh0/ms-coco-dataset

        print("Extracting dataset...")
        with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall('data')

        print("Backing up dataset Zip to Google Drive for next time...")
        os.makedirs(os.path.dirname(DRIVE_ZIP_PATH), exist_ok=True)
        shutil.copy(LOCAL_ZIP_PATH, DRIVE_ZIP_PATH)
        print("Backup complete!")

else:
    print("Data already extracted locally, skipping download.")

# 2. Load JSON files using CONFIG
print("Loading annotations...")
with open(CONFIG["CAPTION_JSON"], 'r') as f:
    caption_data = json.load(f)
with open(CONFIG["INSTANCES_JSON"], 'r') as f:
    instance_data = json.load(f)

# 3. Build Mapping (Filename -> List of Captions)
img_id_to_filename = {img['id']: img['file_name'] for img in caption_data['images']}
mapping = {}
for ann in caption_data['annotations']:
    fname = img_id_to_filename[ann['image_id']]
    if fname not in mapping:
        mapping[fname] = []
    mapping[fname].append(ann['caption'])

# 4. Build Category Mapping (Filename -> Binary Label Vector)
filename_to_cats = {img['file_name']: np.zeros(CONFIG["NUM_CLASSES"]) for img in caption_data['images']}
for ann in instance_data['annotations']:
    fname = img_id_to_filename[ann['image_id']]
    cat_id = ann['category_id'] - 1 # COCO category_ids are 1-80
    if 0 <= cat_id < CONFIG["NUM_CLASSES"]:
        filename_to_cats[fname][cat_id] = 1

print(f"Data loaded successfully. Mapped {len(mapping)} images with auxiliary labels.")

Local data not found. Checking Google Drive...
Dataset not found in Drive. Downloading from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/hariwh0/ms-coco-dataset
License(s): unknown
Resuming from 4212129792 bytes (9468485448 bytes left)...
100% 12.7G/12.7G [07:37<00:00, 20.7MB/s]

Extracting dataset...
Backing up dataset Zip to Google Drive for next time...
Backup complete!
Loading annotations...
Data loaded successfully. Mapped 82783 images with auxiliary labels.


#50% of dataset

In [ ]:
# --- BLOCK 2.4: 50% Dataset Sampling ---
import random

print(f"Original dataset size: {len(mapping)} images")

# 1. Grab all the image names
all_images = list(mapping.keys())

# 2. Shuffle them so we get a random mix of pictures
random.seed(42) # Keeps the random split consistent
random.shuffle(all_images)

# 3. Cut the list exactly in half
half_size = len(all_images) // 2
sampled_images = all_images[:half_size]

# 4. Overwrite our mapping dictionary with only the 50% sample
mapping = {img: mapping[img] for img in sampled_images}

print(f"New 50% dataset size: {len(mapping)} images")

Original dataset size: 82783 images
New 50% dataset size: 41391 images


#Text Preprocessing

In [ ]:
# --- BLOCK 2.5: Text Preprocessing (Cleaning & Tokenization) ---
import re

# 1. Caption Cleaning (Requirement A)
print("Cleaning captions...")
for img_name in mapping:
    clean_captions = []
    for caption in mapping[img_name]:
        # Lowercase and remove special characters
        clean = caption.lower()
        clean = re.sub(r'[^a-z0-9\s]', '', clean)
        # Add start and end sequence tokens
        clean = "startseq " + clean + " endseq"
        clean_captions.append(clean)
    # Replace the raw captions with the cleaned ones
    mapping[img_name] = clean_captions

print("Captions cleaned and formatted with start/end tokens.")

# 2. Vocabulary Construction & Tokenization (Requirement B)
print("Constructing vocabulary...")
# Flatten the dictionary to a single list of all captions
all_captions = [cap for caps in mapping.values() for cap in caps]

# Initialize and fit the tokenizer using our CONFIG setting
tokenizer = Tokenizer(num_words=CONFIG["VOCAB_SIZE"], oov_token="<unk>")
tokenizer.fit_on_texts(all_captions)

print(f"Tokenization complete. Actual vocabulary size found: {len(tokenizer.word_index)}")
print(f"Model will use the top {CONFIG['VOCAB_SIZE']} words as defined in CONFIG.")
print("Sequence preparation (Requirement C) will be handled by the generator during training.")

Cleaning captions...
Captions cleaned and formatted with start/end tokens.
Constructing vocabulary...
Tokenization complete. Actual vocabulary size found: 18406
Model will use the top 5000 words as defined in CONFIG.
Sequence preparation (Requirement C) will be handled by the generator during training.


#Feature Extraction

In [ ]:
# # --- BLOCK 3: Feature Extraction (Resilient & Resume-able) ---
# import random
# # 1. Define the Pre-trained Model
# base_model = InceptionV3(weights='imagenet')
# model = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

# # 2. Setup path
# image_dir = CONFIG["IMG_DIR"]
# features_path = CONFIG["FEATURES_FILE"]

# # 3. Load existing features (with error handling)
# features = {}
# if os.path.exists(features_path):
#     try:
#         print("Found features file. Attempting to load...")
#         # Use allow_pickle=True, but catch errors if file is corrupted
#         features = np.load(features_path, allow_pickle=True).item()
#         print(f"Resuming with {len(features)} existing images.")
#     except Exception as e:
#         print(f"Error loading features file: {e}. File is likely corrupted. Starting fresh.")
#         # If loading fails, delete the corrupted file so it doesn't happen again
#         os.remove(features_path)
#         features = {}
# else:
#     print("No existing features found. Starting from scratch.")

# # 4. Get all images and subset to 50%
# all_image_names = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
# subset_size = int(len(all_image_names) * 0.5)
# random.seed(42)
# selected_images = random.sample(all_image_names, subset_size)

# # 5. Extract and Save
# print(f"Total subset to process: {len(selected_images)} images.")

# for img_name in tqdm(selected_images):
#     # Skip if we already have this image in our dictionary
#     if img_name in features:
#         continue

#     try:
#         img_path = os.path.join(image_dir, img_name)
#         img = load_img(img_path, target_size=(299, 299))
#         img = img_to_array(img)
#         img = np.expand_dims(img, axis=0)
#         img = preprocess_input(img)

#         feature = model.predict(img, verbose=0)
#         features[img_name] = feature

#         # Save every 50 images to keep the file healthy
#         if len(features) % 50 == 0:
#             np.save(features_path, features)

#     except Exception as e:
#         print(f"Error processing {img_name}: {e}")
#         continue

# # Final Save
# np.save(features_path, features)
# print(f"\nExtraction complete! Features saved to '{features_path}'.")

In [ ]:
# --- BLOCK 3: Advanced Feature Extraction (Google Drive Hardwired) ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tqdm import tqdm

print("Building ResNet-50 Spatial CNN (Drive-Aware Mode)...")

# 1. HARDWIRED TO GOOGLE DRIVE: Check here first so we don't duplicate work!
FEATURE_DIR = "/content/drive/MyDrive/attention_features"
if not os.path.exists(FEATURE_DIR):
    os.makedirs(FEATURE_DIR)

# 2. Load ResNet50
base_model = ResNet50(weights='imagenet', include_top=False)
image_model = Model(inputs=base_model.input, outputs=base_model.output)

print(f"Checking images against Drive folder: {FEATURE_DIR}...")
img_list = list(mapping.keys())

for img_name in tqdm(img_list):
    # Save path for this specific image IN GOOGLE DRIVE
    feature_path = os.path.join(FEATURE_DIR, img_name + ".npy")

    # Crash-Proofing: If it's already in Drive, skip it instantly!
    if os.path.exists(feature_path):
        continue

    img_path = os.path.join(CONFIG["IMG_DIR"], img_name)
    try:
        img = load_img(img_path, target_size=(224, 224))
        img = img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)

        feature_grid = image_model.predict(img, verbose=0)
        feature_grid = tf.reshape(feature_grid, (feature_grid.shape[1] * feature_grid.shape[2], feature_grid.shape[3]))

        # Save IMMEDIATELY to Google Drive
        np.save(feature_path, feature_grid.numpy())

    except Exception as e:
        print(f"Skipping {img_name} due to error: {e}")

print("Extraction Complete! All features safely stored in Google Drive.")

Building ResNet-50 Spatial CNN (Drive-Aware Mode)...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Checking images against Drive folder: /content/drive/MyDrive/attention_features...


100%|██████████| 41391/41391 [00:24<00:00, 1669.31it/s]

Extraction Complete! All features safely stored in Google Drive.


In [ ]:
# # --- BLOCK 3.5: Pre-Trained GloVe Word Embeddings ---
# import os
# import numpy as np
# import urllib.request
# import zipfile

# print("Downloading GloVe embeddings (this may take a minute)...")
# glove_zip = "glove.6B.zip"
# if not os.path.exists(glove_zip):
#     urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", glove_zip)
#     with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
#         zip_ref.extractall("glove_data")

# print("Loading GloVe into memory...")
# embeddings_index = {}
# # We use the 200-dimensional GloVe vectors for higher accuracy
# with open(os.path.join("glove_data", "glove.6B.200d.txt"), encoding="utf-8") as f:
#     for line in f:
#         values = line.split()
#         word = values[0]
#         coefs = np.asarray(values[1:], dtype='float32')
#         embeddings_index[word] = coefs

# print(f"Found {len(embeddings_index)} word vectors in GloVe.")

# # Create the embedding matrix for our specific vocabulary
# embedding_dim = 200
# vocab_size = CONFIG["VOCAB_SIZE"]
# embedding_matrix = np.zeros((vocab_size, embedding_dim))

# hits = 0
# misses = 0

# for word, i in tokenizer.word_index.items():
#     if i >= vocab_size:
#         break
#     embedding_vector = embeddings_index.get(word)
#     if embedding_vector is not None:
#         # Words not found in embedding index will be all-zeros.
#         embedding_matrix[i] = embedding_vector
#         hits += 1
#     else:
#         misses += 1

# print(f"Converted: {hits} words ({misses} misses). Ready for Block 4.")

#Model Building

In [ ]:
# # --- BLOCK 4: Model Building (GloVe Upgraded) ---
# from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
# from tensorflow.keras.models import Model

# # 1. Image Branch
# img_in = Input(shape=(2048,), name="image_input")
# img_features = Dropout(0.5)(img_in)
# # Bumped to 512 to match the complex text features
# img_dense = Dense(512, activation='relu')(img_features)

# # 2. Text Branch (Now powered by Stanford GloVe)
# text_in = Input(shape=(CONFIG["MAX_LENGTH"],), name="text_input")

# # Injecting the GloVe matrix here
# text_emb = Embedding(
#     CONFIG["VOCAB_SIZE"],
#     200, # Matches GloVe dimension
#     weights=[embedding_matrix],
#     trainable=False, # Lock it so we don't destroy the pre-trained grammar
#     mask_zero=True
# )(text_in)

# text_dropout = Dropout(0.5)(text_emb)
# # Bumped to 512 for better memory capacity
# text_lstm = LSTM(512)(text_dropout)

# # 3. Captioning Branch (Merging Image + Text)
# decoder = add([img_dense, text_lstm])
# decoder_dense = Dense(512, activation='relu')(decoder)
# cap_out = Dense(CONFIG["VOCAB_SIZE"], activation='softmax', name='caption_output')(decoder_dense)

# # 4. Auxiliary Classification Branch
# class_out = Dense(CONFIG["NUM_CLASSES"], activation='sigmoid', name='classifier_output')(img_dense)

# # 5. Build and Compile
# model = Model(inputs=[img_in, text_in], outputs=[cap_out, class_out])

# model.compile(
#     optimizer='adam',
#     loss={
#         'caption_output': 'categorical_crossentropy',
#         'classifier_output': 'binary_crossentropy'
#     },
#     loss_weights={'caption_output': 1.0, 'classifier_output': 0.2},
#     metrics={'caption_output': 'accuracy', 'classifier_output': 'accuracy'}
# )

# model.summary()

In [ ]:
# --- BLOCK 4: Advanced Architecture (tf_keras Stable Version) ---
import tensorflow as tf
import tf_keras as keras  # Forcing the stable Keras 2 engine!

print("Building Custom Attention Architecture...")

embedding_dim = 256
units = 512
vocab_size = CONFIG["VOCAB_SIZE"]

class CNN_Encoder(keras.Model):
    def __init__(self, embedding_dim):
        super(CNN_Encoder, self).__init__()
        self.fc = keras.layers.Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

class BahdanauAttention(keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = keras.layers.Dense(units)
        self.W2 = keras.layers.Dense(units)
        self.V = keras.layers.Dense(1)

    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class RNN_Decoder(keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super(RNN_Decoder, self).__init__()
        self.units = units
        self.embedding = keras.layers.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(self.units)
        self.lstm_1 = keras.layers.LSTM(self.units, return_sequences=True, return_state=True)
        self.lstm_2 = keras.layers.LSTM(self.units, return_sequences=True, return_state=True)
        self.fc = keras.layers.Dense(vocab_size)

    def call(self, x, features, hidden_1, cell_1, hidden_2, cell_2):
        context_vector, attention_weights = self.attention(features, hidden_2)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        x, h1, c1 = self.lstm_1(x, initial_state=[hidden_1, cell_1])
        x, h2, c2 = self.lstm_2(x, initial_state=[hidden_2, cell_2])
        x = tf.reshape(x, (-1, x.shape[2]))
        x = self.fc(x)
        return x, h1, c1, h2, c2, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units)), tf.zeros((batch_size, self.units))

encoder = CNN_Encoder(embedding_dim)
decoder = RNN_Decoder(embedding_dim, units, vocab_size)

print("Advanced Architecture built and ready!")

Building Custom Attention Architecture...
Advanced Architecture built and ready!


In [ ]:
# --- BLOCK 4.8: Move Features to Fast Local Storage ---
import shutil
import os

print("Copying features from Drive to fast local storage...")
if not os.path.exists("/content/attention_features"):
    # This copies the folder from Drive to Colab's local workspace
    shutil.copytree("/content/drive/MyDrive/attention_features", "/content/attention_features")
    print("✅ Copy complete! Your model will now read at lightning speed.")
else:
    print("✅ Features already in local storage.")

Copying features from Drive to fast local storage...
✅ Features already in local storage.


#Training & Evaluation

In [ ]:
# --- BLOCK 5: Advanced Training Loop (High-Speed Local Version) ---
import os
import time
import numpy as np
import tensorflow as tf
import tf_keras as keras
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import load_img

print("Setting up Custom Training Loop...")

# 1. Optimizer and Loss Function
optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_object = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

# 2. Crash-Proof Checkpoints (STILL DIRECT TO GOOGLE DRIVE - CRASH PROOF!)
checkpoint_path = "/content/drive/MyDrive/AML_Project_Checkpoints"
ckpt = tf.train.Checkpoint(encoder=encoder, decoder=decoder, optimizer=optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=3)

if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint)
    print("Latest checkpoint restored! Resuming training...")
else:
    print("No checkpoint found. Starting fresh.")

# 3. High-Speed Data Generator (READING LOCALLY)
def get_batches(mapping, tokenizer, max_length, batch_size):
    img_feats, targets = [], []
    for img_name, captions in mapping.items():
        # FAST LOCAL PATH
        feature_path = f"attention_features/{img_name}.npy"
        if not os.path.exists(feature_path): continue

        feat = np.load(feature_path)

        for cap in captions:
            seq = tokenizer.texts_to_sequences([cap])[0]
            seq = pad_sequences([seq], maxlen=max_length, padding='post')[0]

            img_feats.append(feat)
            targets.append(seq)

            if len(img_feats) == batch_size:
                yield np.array(img_feats), np.array(targets)
                img_feats, targets = [], []

# 4. Core Training Step
@tf.function
def train_step(img_tensor, target):
    loss = 0
    batch_sz = target.shape[0]

    hidden_1, cell_1 = decoder.reset_state(batch_sz)
    hidden_2, cell_2 = decoder.reset_state(batch_sz)
    dec_input = tf.expand_dims([tokenizer.word_index['startseq']] * batch_sz, 1)

    with tf.GradientTape() as tape:
        features = encoder(img_tensor)
        for i in range(1, target.shape[1]):
            predictions, hidden_1, cell_1, hidden_2, cell_2, _ = decoder(
                dec_input, features, hidden_1, cell_1, hidden_2, cell_2)
            loss += loss_function(target[:, i], predictions)
            dec_input = tf.expand_dims(target[:, i], 1)

    total_loss = (loss / int(target.shape[1]))
    trainable_variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_variables)
    optimizer.apply_gradients(zip(gradients, trainable_variables))

    return loss, total_loss

# 5. The Training Loop
EPOCHS = 10
BATCH_SIZE = 64
STEPS_PER_EPOCH = len(mapping) // BATCH_SIZE

print(f"\n--- Starting Custom Training for {EPOCHS} Epochs ---")
for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0

    batch_generator = get_batches(mapping, tokenizer, CONFIG["MAX_LENGTH"], BATCH_SIZE)

    for batch, (img_tensor, target) in enumerate(batch_generator):
        if batch >= STEPS_PER_EPOCH:
            break

        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss += t_loss

        if batch % 50 == 0:
            print(f'Epoch {epoch+1} Batch {batch} Loss {t_loss.numpy():.4f}')

    ckpt_manager.save()
    print(f'Epoch {epoch+1} Loss {total_loss/STEPS_PER_EPOCH:.4f}')
    print(f'Time taken for 1 epoch {time.time() - start:.2f} sec\n')

# 6. Evaluation Function (READING LOCALLY)
def evaluate_attention(img_name):
    # FAST LOCAL PATH
    feature_path = f"attention_features/{img_name}.npy"
    photo_features = np.load(feature_path)

    hidden_1, cell_1 = decoder.reset_state(batch_size=1)
    hidden_2, cell_2 = decoder.reset_state(batch_size=1)

    dec_input = tf.expand_dims([tokenizer.word_index['startseq']], 0)
    features = encoder(tf.expand_dims(photo_features, 0))

    result = []
    for i in range(CONFIG["MAX_LENGTH"]):
        predictions, hidden_1, cell_1, hidden_2, cell_2, attention_weights = decoder(
            dec_input, features, hidden_1, cell_1, hidden_2, cell_2)

        predicted_id = tf.argmax(predictions[0]).numpy()
        word = tokenizer.index_word.get(predicted_id)

        if word is None or word == 'endseq':
            break

        result.append(word)
        dec_input = tf.expand_dims([predicted_id], 0)

    return ' '.join(result)

# 7. Visual Output Test
print("\n--- Visual Output Test (Attention) ---")
try:
    sample_img_name = list(mapping.keys())[5]
    caption = evaluate_attention(sample_img_name)

    plt.figure(figsize=(8, 8))
    plt.imshow(load_img(os.path.join(CONFIG["IMG_DIR"], sample_img_name)))
    plt.title(f"Attention Output: {caption.capitalize()}", fontsize=14)
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Could not display sample image. Error: {e}")

Setting up Custom Training Loop...
Latest checkpoint restored! Resuming training...

--- Starting Custom Training for 10 Epochs ---
Epoch 1 Batch 0 Loss 1.0866
Epoch 1 Batch 50 Loss 1.0835
Epoch 1 Batch 100 Loss 1.0114
Epoch 1 Batch 150 Loss 1.0256
Epoch 1 Batch 200 Loss 1.1490
Epoch 1 Batch 250 Loss 0.9767
Epoch 1 Batch 300 Loss 1.1502


KeyboardInterrupt: 

In [ ]:
# --- BLOCK 6: The Ultimate Google Drive Backup ---
from google.colab import drive
import shutil
import os

print("Training finished! Initiating emergency backup sequence...")

# 1. Mount Google Drive (It might pop up a window asking for permission)
drive.mount('/content/drive')

# 2. Create a secure folder in your Drive
backup_folder = "/content/drive/MyDrive/AML_Project_Backup"
if not os.path.exists(backup_folder):
    os.makedirs(backup_folder)

# 3. Backup the Checkpoints (The Model's Brain!)
print("Backing up model checkpoints...")
shutil.copytree("attention_checkpoints", os.path.join(backup_folder, "attention_checkpoints"), dirs_exist_ok=True)

# 4. Backup the Extracted Features (So you never wait 2 hours again!)
print("Backing up spatial features...")
shutil.copytree("attention_features", os.path.join(backup_folder, "attention_features"), dirs_exist_ok=True)

print(f"BACKUP COMPLETE! All files safely stored in your Google Drive at: {backup_folder}")

In [ ]:
3.# # --- BLOCK 5: Training & Evaluation (GloVe + Beam Search Upgraded) ---

# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tensorflow.keras.utils import to_categorical
# from tensorflow.keras.preprocessing.image import load_img
# from nltk.translate.bleu_score import corpus_bleu

# print("Preparing batched data generator and callbacks...")

# # 1. Batched Data Generator (Keras 3 Compliant with Dictionaries)
# def data_generator(mapping, features, tokenizer, max_length, cat_map, batch_size):
#     X1, X2, y_cap, y_class = list(), list(), list(), list()
#     n = 0
#     while True:
#         for img_name, captions in mapping.items():
#             if img_name not in features:
#                 continue

#             photo = features[img_name][0]
#             cat_label = cat_map.get(img_name, np.zeros(CONFIG["NUM_CLASSES"]))

#             for caption in captions:
#                 seq = tokenizer.texts_to_sequences([caption])[0]
#                 for i in range(1, len(seq)):
#                     in_seq = pad_sequences([seq[:i]], maxlen=max_length)[0]
#                     out_word = to_categorical([seq[i]], num_classes=CONFIG["VOCAB_SIZE"])[0]

#                     X1.append(photo)
#                     X2.append(in_seq)
#                     y_cap.append(out_word)
#                     y_class.append(cat_label)
#                     n += 1

#                     if n == batch_size:
#                         x_batch = {
#                             "image_input": np.array(X1),
#                             "text_input": np.array(X2)
#                         }
#                         y_batch = {
#                             "caption_output": np.array(y_cap),
#                             "classifier_output": np.array(y_class)
#                         }
#                         yield (x_batch, y_batch)
#                         X1, X2, y_cap, y_class = list(), list(), list(), list()
#                         n = 0

# # 2. Crash-Proof Setup & Pro Callbacks
# checkpoint_path = CONFIG["MODEL_SAVE_PATH"]

# if os.path.exists(checkpoint_path):
#     print("Checkpoint found! Loading saved weights to resume training...")
#     model.load_weights(checkpoint_path)
# else:
#     print("No previous checkpoint found. Starting fresh.")

# checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='loss', verbose=1)
# early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True, verbose=1)
# reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.2, patience=3, min_lr=0.0001, verbose=1)

# # 3. Training (500 steps * 64 batch size)
# print(f"\n--- Starting training for {CONFIG['EPOCHS']} epochs ---")
# history = model.fit(
#     data_generator(mapping, features, tokenizer, CONFIG["MAX_LENGTH"], filename_to_cats, CONFIG["BATCH_SIZE"]),
#     steps_per_epoch=500,
#     epochs=CONFIG["EPOCHS"],
#     callbacks=[checkpoint, early_stop, reduce_lr],
#     verbose=1
# )

# # 4. Helper Functions for Generation (Basic vs Beam)
# def generate_caption_greedy(photo):
#     """The original fast, basic guesser."""
#     in_text = 'startseq'
#     for i in range(CONFIG["MAX_LENGTH"]):
#         sequence = tokenizer.texts_to_sequences([in_text])[0]
#         sequence = pad_sequences([sequence], maxlen=CONFIG["MAX_LENGTH"])

#         x_pred = {"image_input": photo, "text_input": sequence}
#         yhat = model.predict(x_pred, verbose=0)[0][0]
#         yhat = np.argmax(yhat)

#         word = tokenizer.index_word.get(yhat)
#         if word is None: break
#         in_text += ' ' + word
#         if word == 'endseq': break

#     return in_text.replace('startseq ', '').replace(' endseq', '')

# def generate_caption_beam_search(photo, beam_width=3):
#     """The upgraded pro-level generator."""
#     start_seq = [tokenizer.word_index['startseq']]
#     sequences = [[start_seq, 0.0]]

#     for i in range(CONFIG["MAX_LENGTH"]):
#         all_candidates = list()
#         for seq, score in sequences:
#             if seq[-1] == tokenizer.word_index.get('endseq'):
#                 all_candidates.append([seq, score])
#                 continue

#             pad_seq = pad_sequences([seq], maxlen=CONFIG["MAX_LENGTH"])[0]
#             x_pred = {"image_input": photo, "text_input": np.array([pad_seq])}
#             preds = model.predict(x_pred, verbose=0)[0][0]

#             top_words = np.argsort(preds)[-beam_width:]
#             for word_index in top_words:
#                 candidate_seq = seq + [word_index]
#                 candidate_score = score - np.log(preds[word_index] + 1e-10)
#                 all_candidates.append([candidate_seq, candidate_score])

#         ordered = sorted(all_candidates, key=lambda tup: tup[1])
#         sequences = ordered[:beam_width]

#     best_seq = sequences[0][0]

#     final_caption = []
#     for word_index in best_seq:
#         word = tokenizer.index_word.get(word_index)
#         if word == 'startseq': continue
#         if word == 'endseq': break
#         if word: final_caption.append(word)

#     return ' '.join(final_caption)

# # 5. Quantitative Evaluation (BLEU Scores)
# print("\n--- Evaluating Model (BLEU Scores) ---")
# print("Calculating scores on a test subset of 50 images to conserve memory...")
# # Note: We use Greedy here to calculate BLEU quickly. Beam Search on 50 images takes much longer.
# actual, predicted = list(), list()
# test_count = 0

# for img_name, captions in mapping.items():
#     if img_name not in features: continue
#     if test_count >= 50: break

#     photo_input = np.array([features[img_name][0]])
#     yhat = generate_caption_greedy(photo_input)

#     predicted.append(yhat.split())
#     actual_captions = [cap.replace('startseq ', '').replace(' endseq', '').split() for cap in captions]
#     actual.append(actual_captions)
#     test_count += 1

# print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
# print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

# # 6. Qualitative Evaluation (Visual Test)
# print("\n--- Visual Output Test (Greedy vs. Beam Search) ---")
# try:
#     img_name = list(features.keys())[3] # Testing a new image
#     photo = np.array([features[img_name][0]])

#     # Compare the two methods
#     greedy_caption = generate_caption_greedy(photo)
#     beam_caption = generate_caption_beam_search(photo, beam_width=3)

#     plt.figure(figsize=(8, 8))
#     plt.imshow(load_img(os.path.join(CONFIG["IMG_DIR"], img_name)))
#     plt.title(f"Basic Guess: {greedy_caption.capitalize()}\nBeam Search: {beam_caption.capitalize()}", fontsize=12)
#     plt.axis('off')
#     plt.show()
# except Exception as e:
#     print(f"Could not display sample image. Error: {e}")

In [ ]:
1.
# # --- BLOCK 5: Training & Evaluation (Final Batched Version) ---

# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from tensorflow.keras.callbacks import ModelCheckpoint
# from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tensorflow.keras.utils import to_categorical
# from tensorflow.keras.preprocessing.image import load_img
# from nltk.translate.bleu_score import corpus_bleu

# print("Preparing batched data generator and checkpoints...")

# # 1. Batched Data Generator (Keras 3 Compliant with Dictionaries)
# def data_generator(mapping, features, tokenizer, max_length, cat_map, batch_size):
#     X1, X2, y_cap, y_class = list(), list(), list(), list()
#     n = 0
#     while True:
#         for img_name, captions in mapping.items():
#             if img_name not in features:
#                 continue

#             photo = features[img_name][0]
#             cat_label = cat_map.get(img_name, np.zeros(CONFIG["NUM_CLASSES"]))

#             for caption in captions:
#                 seq = tokenizer.texts_to_sequences([caption])[0]
#                 for i in range(1, len(seq)):
#                     in_seq = pad_sequences([seq[:i]], maxlen=max_length)[0]
#                     out_word = to_categorical([seq[i]], num_classes=CONFIG["VOCAB_SIZE"])[0]

#                     # Accumulate data into the batch lists
#                     X1.append(photo)
#                     X2.append(in_seq)
#                     y_cap.append(out_word)
#                     y_class.append(cat_label)
#                     n += 1

#                     # Once we hit the batch size, yield the dictionaries and reset
#                     if n == batch_size:
#                         x_batch = {
#                             "image_input": np.array(X1),
#                             "text_input": np.array(X2)
#                         }
#                         y_batch = {
#                             "caption_output": np.array(y_cap),
#                             "classifier_output": np.array(y_class)
#                         }
#                         yield (x_batch, y_batch)
#                         X1, X2, y_cap, y_class = list(), list(), list(), list()
#                         n = 0

# # 2. Crash-Proof Setup
# checkpoint_path = CONFIG["MODEL_SAVE_PATH"]

# if os.path.exists(checkpoint_path):
#     print("Checkpoint found! Loading saved weights to resume training...")
#     model.load_weights(checkpoint_path)
# else:
#     print("No previous checkpoint found. Starting fresh.")

# checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='loss', verbose=1)

# # 3. Training
# # instead of 200 steps * 1 = 200 per epoch :( .
# # 200 steps * 64 batch size = 12,800 words seen per epoch!
# print(f"\n--- Starting training for {CONFIG['EPOCHS']} epochs ---")
# history = model.fit(
#     data_generator(mapping, features, tokenizer, CONFIG["MAX_LENGTH"], filename_to_cats, CONFIG["BATCH_SIZE"]),
#     steps_per_epoch=200,
#     epochs=CONFIG["EPOCHS"],
#     callbacks=[checkpoint],
#     verbose=1
# )

# # 4. Helper Function for Generation
# def generate_caption(photo):
#     in_text = 'startseq'
#     for i in range(CONFIG["MAX_LENGTH"]):
#         sequence = tokenizer.texts_to_sequences([in_text])[0]
#         sequence = pad_sequences([sequence], maxlen=CONFIG["MAX_LENGTH"])

#         # We pass a dictionary to predict() for Keras 3 compatibility
#         x_pred = {
#             "image_input": photo,
#             "text_input": sequence
#         }

#         yhat = model.predict(x_pred, verbose=0)[0]
#         yhat = np.argmax(yhat)

#         word = tokenizer.index_word.get(yhat)
#         if word is None:
#             break

#         in_text += ' ' + word
#         if word == 'endseq':
#             break

#     return in_text.replace('startseq ', '').replace(' endseq', '')

# # 5. Quantitative Evaluation (BLEU Scores)
# print("\n--- Evaluating Model (BLEU Scores) ---")
# print("Calculating scores on a test subset of 50 images to conserve memory...")

# actual, predicted = list(), list()
# test_count = 0

# for img_name, captions in mapping.items():
#     if img_name not in features:
#         continue
#     if test_count >= 50:
#         break

#     # Predict expects shape (1, 2048)
#     photo_input = np.array([features[img_name][0]])
#     yhat = generate_caption(photo_input)

#     predicted.append(yhat.split())
#     actual_captions = [cap.replace('startseq ', '').replace(' endseq', '').split() for cap in captions]
#     actual.append(actual_captions)

#     test_count += 1

# print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
# print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

# # 6. Qualitative Evaluation (Visual Test)
# print("\n--- Visual Output Test ---")
# try:
#     # Pick the second image to test (so it's different from the Wii remote one!)
#     img_name = list(features.keys())[1]
#     photo = np.array([features[img_name][0]])
#     caption = generate_caption(photo)

#     plt.figure(figsize=(8, 8))
#     plt.imshow(load_img(os.path.join(CONFIG["IMG_DIR"], img_name)))
#     plt.title(f"Generated Caption: {caption.capitalize()}", fontsize=14)
#     plt.axis('off')
#     plt.show()
# except Exception as e:
#     print(f"Could not display sample image. Error: {e}")

In [ ]:
2.# # --- BLOCK 5: Training & Evaluation (Pro Overnight Setup) ---

# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tensorflow.keras.utils import to_categorical
# from tensorflow.keras.preprocessing.image import load_img
# from nltk.translate.bleu_score import corpus_bleu

# print("Preparing batched data generator and callbacks...")

# # 1. Batched Data Generator (Keras 3 Compliant with Dictionaries)
# def data_generator(mapping, features, tokenizer, max_length, cat_map, batch_size):
#     X1, X2, y_cap, y_class = list(), list(), list(), list()
#     n = 0
#     while True:
#         for img_name, captions in mapping.items():
#             if img_name not in features:
#                 continue

#             photo = features[img_name][0]
#             cat_label = cat_map.get(img_name, np.zeros(CONFIG["NUM_CLASSES"]))

#             for caption in captions:
#                 seq = tokenizer.texts_to_sequences([caption])[0]
#                 for i in range(1, len(seq)):
#                     in_seq = pad_sequences([seq[:i]], maxlen=max_length)[0]
#                     out_word = to_categorical([seq[i]], num_classes=CONFIG["VOCAB_SIZE"])[0]

#                     # Accumulate data into the batch lists
#                     X1.append(photo)
#                     X2.append(in_seq)
#                     y_cap.append(out_word)
#                     y_class.append(cat_label)
#                     n += 1

#                     # Once we hit the batch size, yield the dictionaries and reset
#                     if n == batch_size:
#                         x_batch = {
#                             "image_input": np.array(X1),
#                             "text_input": np.array(X2)
#                         }
#                         y_batch = {
#                             "caption_output": np.array(y_cap),
#                             "classifier_output": np.array(y_class)
#                         }
#                         yield (x_batch, y_batch)
#                         X1, X2, y_cap, y_class = list(), list(), list(), list()
#                         n = 0

# # 2. Crash-Proof Setup & Pro Callbacks
# checkpoint_path = CONFIG["MODEL_SAVE_PATH"]

# if os.path.exists(checkpoint_path):
#     print("Checkpoint found! Loading saved weights to resume training...")
#     model.load_weights(checkpoint_path)
# else:
#     print("No previous checkpoint found. Starting fresh.")

# # The 3 "Night-Shift Managers"
# checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='loss', verbose=1)
# early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True, verbose=1)
# reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.2, patience=3, min_lr=0.0001, verbose=1)

# # 3. Training
# # 500 steps * 64 batch size = 32,000 words seen per epoch!
# print(f"\n--- Starting training for {CONFIG['EPOCHS']} epochs ---")
# history = model.fit(
#     data_generator(mapping, features, tokenizer, CONFIG["MAX_LENGTH"], filename_to_cats, CONFIG["BATCH_SIZE"]),
#     steps_per_epoch=500,
#     epochs=CONFIG["EPOCHS"],
#     callbacks=[checkpoint, early_stop, reduce_lr],
#     verbose=1
# )

# # 4. Helper Function for Generation
# def generate_caption(photo):
#     in_text = 'startseq'
#     for i in range(CONFIG["MAX_LENGTH"]):
#         sequence = tokenizer.texts_to_sequences([in_text])[0]
#         sequence = pad_sequences([sequence], maxlen=CONFIG["MAX_LENGTH"])

#         # We pass a dictionary to predict() for Keras 3 compatibility
#         x_pred = {
#             "image_input": photo,
#             "text_input": sequence
#         }

#         yhat = model.predict(x_pred, verbose=0)[0]
#         yhat = np.argmax(yhat)

#         word = tokenizer.index_word.get(yhat)
#         if word is None:
#             break

#         in_text += ' ' + word
#         if word == 'endseq':
#             break

#     return in_text.replace('startseq ', '').replace(' endseq', '')

# # 5. Quantitative Evaluation (BLEU Scores)
# print("\n--- Evaluating Model (BLEU Scores) ---")
# print("Calculating scores on a test subset of 50 images to conserve memory...")

# actual, predicted = list(), list()
# test_count = 0

# for img_name, captions in mapping.items():
#     if img_name not in features:
#         continue
#     if test_count >= 50:
#         break

#     photo_input = np.array([features[img_name][0]])
#     yhat = generate_caption(photo_input)

#     predicted.append(yhat.split())
#     actual_captions = [cap.replace('startseq ', '').replace(' endseq', '').split() for cap in captions]
#     actual.append(actual_captions)

#     test_count += 1

# print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
# print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

# # 6. Qualitative Evaluation (Visual Test)
# print("\n--- Visual Output Test ---")
# try:
#     # Pick an image to test (Index 2 to keep getting fresh examples)
#     img_name = list(features.keys())[2]
#     photo = np.array([features[img_name][0]])
#     caption = generate_caption(photo)

#     plt.figure(figsize=(8, 8))
#     plt.imshow(load_img(os.path.join(CONFIG["IMG_DIR"], img_name)))
#     plt.title(f"Generated Caption: {caption.capitalize()}", fontsize=14)
#     plt.axis('off')
#     plt.show()
# except Exception as e:
#     print(f"Could not display sample image. Error: {e}")